# Architecture Ablation Study

Tests four architectural changes made to close gaps vs. the reference paper
(https://doi.org/10.3390/electronics12183960).

| # | Change | Variants tested |
|---|---|---|
| 1 | Window length | 40 vs **64** (new default) |
| 2 | Transformer depth | 2 vs **6** layers (new default) |
| 3 | LSTM MLP head | single Linear vs **Linear+ReLU+Dropout+BN+Linear** |
| 4 | Momentum pre-filter | effect on test-set signal quality |

Each experiment trains two variants on identical data splits and compares test AUC
with 95% bootstrap confidence intervals.  Non-overlapping CIs = significant difference.

## 0. Setup

In [ ]:
from sentiment.log import setup_logging
setup_logging()

In [ ]:
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sentiment.sources.cache import MarketDataCache
from sentiment.sources.fundamental import FundamentalCache
from sentiment.features.technical import TechnicalFactors
from sentiment.features.dataloader import build_dataset, make_loaders
from sentiment.features.screening import momentum_slope
from sentiment.model import (
    SentimentLSTM, SentimentTransformer,
    train_model, evaluate, bootstrap_evaluate, collect_predictions,
)

TICKER = "AAPL"
YEAR   = 2025
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED   = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"device: {DEVICE}")

In [ ]:
cache = MarketDataCache()
df = cache.load(TICKER, YEAR)
if df is None or df.empty:
    raise RuntimeError(f"No market data for {TICKER} {YEAR} â€” run market.ipynb first")

fund_cache = FundamentalCache()
fund_df = fund_cache.load_df(TICKER)   # DatetimeIndex'd DataFrame or None

technical = TechnicalFactors()

print(f"market rows : {len(df)}")
print(f"date range  : {df.index[0].date()} -> {df.index[-1].date()}")
print(f"fund rows   : {len(fund_df) if fund_df is not None else 'no data'}")

---
## Experiment 1 Window length: 40 vs 64

Same LSTM architecture, same train/val/test split fractions.  
Longer windows give the model more context per prediction but reduce the number
of training windows from the same data.

In [ ]:
ds_40 = build_dataset(df, technical, ticker=TICKER, window=40,  fundamental_df=fund_df)
ds_64 = build_dataset(df, technical, ticker=TICKER, window=64,  fundamental_df=fund_df)

N_FUND = ds_64["X_fundamental"].shape[1]

# Class balance from the window=64 training split (used for pos_weight)
N64 = len(ds_64["y"])
val_start_64 = int(int(N64 * 0.8) * 0.9)
y_train_64 = ds_64["y"][:val_start_64]
n_pos = int(y_train_64.sum())
n_neg = len(y_train_64) - n_pos
pos_weight = n_neg / n_pos if n_pos > 0 else 1.0

print(f"window=40 -> {len(ds_40['y'])} windows")
print(f"window=64 -> {len(ds_64['y'])} windows")
print(f"n_fund    : {N_FUND}")
print(f"train balance â€” pos: {n_pos} ({n_pos/len(y_train_64):.1%})  neg: {n_neg} ({n_neg/len(y_train_64):.1%})")
print(f"pos_weight: {pos_weight:.3f}")

In [ ]:
train_40, val_40, test_40, _, _ = make_loaders(ds_40, batch_size=32)
train_64, val_64, test_64, _, _ = make_loaders(ds_64, batch_size=32)

lstm_w40 = SentimentLSTM(n_fundamentals=N_FUND)
lstm_w64 = SentimentLSTM(n_fundamentals=N_FUND)

print("Training LSTM  window=40 ...")
h_w40 = train_model(lstm_w40, train_40, val_40, n_epochs=100, patience=15,
                    device=DEVICE, seed=SEED, pos_weight=pos_weight)
print(f"  best epoch {h_w40['best_epoch']}  val_auc={h_w40['best_val_auc']:.4f}")

print("Training LSTM  window=64 ...")
h_w64 = train_model(lstm_w64, train_64, val_64, n_epochs=100, patience=15,
                    device=DEVICE, seed=SEED, pos_weight=pos_weight)
print(f"  best epoch {h_w64['best_epoch']}  val_auc={h_w64['best_val_auc']:.4f}")

In [ ]:
bs_w40 = bootstrap_evaluate(lstm_w40, test_40, device=DEVICE, n_bootstrap=1000, seed=SEED)
bs_w64 = bootstrap_evaluate(lstm_w64, test_64, device=DEVICE, n_bootstrap=1000, seed=SEED)

def fmt_ci(r, m):
    return f"{r[f'{m}_mean']:.4f}  [{r[f'{m}_ci_low']:.4f} - {r[f'{m}_ci_high']:.4f}]"

metrics = ["auc", "accuracy", "precision", "recall"]
w_df = pd.DataFrame(
    {m: {"LSTM window=40": fmt_ci(bs_w40, m), "LSTM window=64": fmt_ci(bs_w64, m)} for m in metrics}
).T
w_df.index.name = "metric"
print("95% bootstrap CIs -- window ablation")
display(w_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, metric, label in [
    (axes[0], "val_auc",  "Validation AUC"),
    (axes[1], "val_loss", "Validation Loss"),
]:
    ax.plot(h_w40[metric], label="window=40", linewidth=2)
    ax.plot(h_w64[metric], label="window=64", linewidth=2, linestyle="--")
    ax.axvline(h_w40["best_epoch"] - 1, color="C0", alpha=0.3, linestyle=":")
    ax.axvline(h_w64["best_epoch"] - 1, color="C1", alpha=0.3, linestyle=":")
    ax.set_title(f"Exp 1 -- {label}")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Experiment 2 — Transformer depth grid search

Trains a `SentimentTransformer` for each depth in `_DEPTH_GRID` on the window=64
split.  Current default is **6 layers** (matching the reference paper) — this
experiment checks whether a different depth performs better on this dataset.

Each variant uses the same train/val/test split and is evaluated with 95%
bootstrap confidence intervals.  Non-overlapping CIs = significant difference.

```
# Future enhancements
# B: Replace grid search with Bayesian optimisation (optuna) — co-optimise n_layers,
#    d_model, nhead, and dropout simultaneously for more sample-efficient search.
# C: Learning-curve diagnostic — train one deep model (e.g. n_layers=10), log
#    per-layer gradient norms and val AUC per epoch to find the depth ceiling
#    empirically (single run, tells you *why* a depth works or doesn't).
```

In [ ]:
_DEPTH_GRID = [2, 4, 6, 8, 10]

tf_models: dict[int, SentimentTransformer] = {}
h_tf_dict: dict[int, dict] = {}

for n_layers in _DEPTH_GRID:
    tf_models[n_layers] = SentimentTransformer(n_fundamentals=N_FUND, n_layers=n_layers)
    print(f"Training Transformer  n_layers={n_layers} ...")
    h_tf_dict[n_layers] = train_model(
        tf_models[n_layers], train_64, val_64,
        n_epochs=100, patience=15,
        device=DEVICE, seed=SEED, pos_weight=pos_weight,
    )
    print(f"  best epoch {h_tf_dict[n_layers]['best_epoch']}  "
          f"val_auc={h_tf_dict[n_layers]['best_val_auc']:.4f}")

# Convenience aliases used by the summary cell
tf_2L, tf_6L = tf_models[2], tf_models[6]
h_tf2, h_tf6 = h_tf_dict[2], h_tf_dict[6]

In [ ]:
bs_tf_dict = {
    n: bootstrap_evaluate(tf_models[n], test_64, device=DEVICE, n_bootstrap=1000, seed=SEED)
    for n in _DEPTH_GRID
}

depth_df = pd.DataFrame(
    {
        m: {
            f"n_layers={n}{'  *' if n == 6 else ''}": fmt_ci(bs_tf_dict[n], m)
            for n in _DEPTH_GRID
        }
        for m in metrics
    }
).T
depth_df.index.name = "metric"
print("95% bootstrap CIs -- transformer depth grid  (* = current default n_layers=6)")
display(depth_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = plt.cm.viridis(np.linspace(0, 0.85, len(_DEPTH_GRID)))

for ax, metric, label in [
    (axes[0], "val_auc",  "Validation AUC"),
    (axes[1], "val_loss", "Validation Loss"),
]:
    for i, n_layers in enumerate(_DEPTH_GRID):
        ls = "--" if n_layers == 6 else "-"
        ax.plot(
            h_tf_dict[n_layers][metric],
            label=f"n_layers={n_layers}{'  (default)' if n_layers == 6 else ''}",
            linewidth=2, linestyle=ls, color=colors[i],
        )
        ax.axvline(
            h_tf_dict[n_layers]["best_epoch"] - 1,
            color=colors[i], alpha=0.2, linestyle=":",
        )
    ax.set_title(f"Exp 2 -- {label}")
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Experiment 3 â€” LSTM MLP head: single Linear vs new Sequential

The new `classifier` is `Linear -> ReLU -> Dropout -> BatchNorm1d -> Linear`.  
To get a fair baseline we reconstruct the old single-linear head inline using the
same `SentimentLSTM` but overriding only the classifier after init so all other
hyperparameters are identical.

In [ ]:
import torch.nn as nn

HIDDEN = 32

# New MLP head (default, already in SentimentLSTM)
lstm_mlp = SentimentLSTM(n_fundamentals=N_FUND, hidden_size=HIDDEN)

# Baseline: single Linear head — derive input dim from the model to stay in sync
# with any future changes to n_fundamentals or n_sentiment_probs.
lstm_lin = SentimentLSTM(n_fundamentals=N_FUND, hidden_size=HIDDEN)
classifier_in = lstm_lin.classifier[0].in_features
lstm_lin.classifier = nn.Linear(classifier_in, 1)   # override with old single-layer head

print("Training LSTM  single Linear head ...")
h_lin = train_model(lstm_lin, train_64, val_64, n_epochs=100, patience=15,
                    device=DEVICE, seed=SEED, pos_weight=pos_weight)
print(f"  best epoch {h_lin['best_epoch']}  val_auc={h_lin['best_val_auc']:.4f}")

print("Training LSTM  MLP head (new) ...")
h_mlp = train_model(lstm_mlp, train_64, val_64, n_epochs=100, patience=15,
                    device=DEVICE, seed=SEED, pos_weight=pos_weight)
print(f"  best epoch {h_mlp['best_epoch']}  val_auc={h_mlp['best_val_auc']:.4f}")

In [ ]:
bs_lin = bootstrap_evaluate(lstm_lin, test_64, device=DEVICE, n_bootstrap=1000, seed=SEED)
bs_mlp = bootstrap_evaluate(lstm_mlp, test_64, device=DEVICE, n_bootstrap=1000, seed=SEED)

head_df = pd.DataFrame(
    {m: {"LSTM single Linear": fmt_ci(bs_lin, m), "LSTM MLP head (new)": fmt_ci(bs_mlp, m)} for m in metrics}
).T
head_df.index.name = "metric"
print("95% bootstrap CIs -- LSTM head ablation")
display(head_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, metric, label in [
    (axes[0], "val_auc",  "Validation AUC"),
    (axes[1], "val_loss", "Validation Loss"),
]:
    ax.plot(h_lin[metric], label="single Linear", linewidth=2)
    ax.plot(h_mlp[metric], label="MLP head (new)", linewidth=2, linestyle="--")
    ax.axvline(h_lin["best_epoch"] - 1, color="C0", alpha=0.3, linestyle=":")
    ax.axvline(h_mlp["best_epoch"] - 1, color="C1", alpha=0.3, linestyle=":")
    ax.set_title(f"Exp 3 -- {label}")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Experiment 4 â€” Momentum pre-filter (20-day slope gate)

`momentum_slope()` fits a linear regression on the last 20 closes.  Positive slope
= uptrend; negative = downtrend.  The filter gates which model predictions to *act on*
at inference -- it is **not** applied during training.

We use the best model from Experiment 1 (window=64 LSTM) and compare:
- **All test windows** -- raw model AUC
- **Uptrend windows only** -- AUC on the subset where `slope > 0`
- **Downtrend windows** -- AUC on the subset where `slope <= 0`

In [ ]:
# Collect raw close prices aligned with the test window end dates
N      = len(ds_64["y"])
test_start = int(N * (1 - 0.2))   # mirrors make_loaders test_frac=0.2
test_dates = ds_64["dates"][test_start:]

close_series = df["close"]

slopes = []
for end_date in test_dates:
    end_ts = pd.Timestamp(end_date)
    loc = close_series.index.get_indexer([end_ts], method="pad")[0]
    if loc < 20:
        slopes.append(0.0)
    else:
        window_closes = close_series.iloc[loc - 19 : loc + 1]
        slopes.append(momentum_slope(window_closes, window=20))

slopes = np.array(slopes)
uptrend_mask = slopes > 0

print(f"Test windows        : {len(slopes)}")
print(f"Uptrend (slope > 0) : {uptrend_mask.sum()}  ({uptrend_mask.mean():.1%})")
print(f"Downtrend (slope<=0): {(~uptrend_mask).sum()}  ({(~uptrend_mask).mean():.1%})")

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score

# Collect all test predictions in one pass using the public API
probs_all, targets_all = collect_predictions(lstm_w64, test_64, DEVICE)

def score(mask, label):
    p, t = probs_all[mask], targets_all[mask]
    preds = (p >= 0.5).astype(int)
    return {
        "subset": label,
        "n": int(mask.sum()),
        "auc":       round(float(roc_auc_score(t, p)),                        4) if len(np.unique(t)) > 1 else None,
        "accuracy":  round(float(accuracy_score(t, preds)),                   4),
        "precision": round(float(precision_score(t, preds, zero_division=0)), 4),
        "recall":    round(float(recall_score(t, preds, zero_division=0)),    4),
    }

all_mask = np.ones(len(probs_all), dtype=bool)
rows = [
    score(all_mask,      "All windows"),
    score(uptrend_mask,  "Uptrend only (slope > 0)"),
    score(~uptrend_mask, "Downtrend (slope <= 0)"),
]
filter_df = pd.DataFrame(rows).set_index("subset")
print("Momentum filter effect on test predictions (LSTM window=64)")
display(filter_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(slopes, bins=30, edgecolor="white", linewidth=0.5)
axes[0].axvline(0, color="red", linestyle="--", label="slope=0 gate")
axes[0].set_title("20-day momentum slope distribution (test set)")
axes[0].set_xlabel("slope (price-units / day)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(probs_all[uptrend_mask],  bins=20, alpha=0.6, label="uptrend",   edgecolor="white")
axes[1].hist(probs_all[~uptrend_mask], bins=20, alpha=0.6, label="downtrend", edgecolor="white")
axes[1].axvline(0.5, color="red", linestyle="--", label="decision threshold")
axes[1].set_title("Model probability by trend regime")
axes[1].set_xlabel("P(up)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Summary -- All experiments

Point-estimate AUC on the test set for every variant.  
Use the bootstrap CIs above to judge significance.

In [ ]:
def point_auc(model, loader):
    return round(evaluate(model, loader, device=DEVICE)["auc"], 4)

rows = [
    {"experiment": "Window",   "variant": "LSTM window=40",       "test_auc": point_auc(lstm_w40, test_40), "best_epoch": h_w40["best_epoch"]},
    {"experiment": "Window",   "variant": "LSTM window=64 (new)", "test_auc": point_auc(lstm_w64, test_64), "best_epoch": h_w64["best_epoch"]},
    *[
        {
            "experiment": "Transformer depth",
            "variant": f"Transformer n_layers={n}{'  (default)' if n == 6 else ''}",
            "test_auc": point_auc(tf_models[n], test_64),
            "best_epoch": h_tf_dict[n]["best_epoch"],
        }
        for n in _DEPTH_GRID
    ],
    {"experiment": "LSTM head", "variant": "LSTM single Linear",   "test_auc": point_auc(lstm_lin, test_64), "best_epoch": h_lin["best_epoch"]},
    {"experiment": "LSTM head", "variant": "LSTM MLP head (new)",  "test_auc": point_auc(lstm_mlp, test_64), "best_epoch": h_mlp["best_epoch"]},
]

summary = pd.DataFrame(rows).set_index(["experiment", "variant"])
display(summary)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 0.55 * len(summary) + 1.5))
labels = [v for _, v in summary.index]
aucs   = summary["test_auc"].values

# Color: orange (C1) = new/current default, blue (C0) = baseline variant
def _bar_color(variant: str) -> str:
    if "new" in variant or "default" in variant:
        return "C1"
    return "C0"

colors_bar = [_bar_color(v) for _, v in summary.index]
bars = ax.barh(labels, aucs, color=colors_bar, edgecolor="white")
ax.axvline(0.5, color="red", linestyle="--", linewidth=1, label="random baseline")
for bar, auc in zip(bars, aucs):
    ax.text(auc + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{auc:.4f}", va="center", fontsize=9)
ax.set_xlabel("Test AUC")
ax.set_title("Ablation study -- test AUC by variant (orange = new/current default)")
ax.legend()
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()